In [ ]:
# pick a GPU runtime (T4 is common on free Colab).
# !pip install -q transformers datasets evaluate jiwer soundfile accelerate peft huggingface_hub safetensors
# !pip install --upgrade transformers datasets peft evaluate accelerate bitsandbytes



In [ ]:
# !pip install datasets pandas pyarrow
# !pip install torchcodec
# !pip install evaluate
# !pip install jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.7 MB/s eta 0:00:00


In [ ]:
# !wget https://commonvoice.mozilla.org/datasets -O cv_twi.html


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # follow prompt to paste your HF token


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, PeftModel
import evaluate


In [ ]:
MODEL_NAME = "dkt-py-bot/Whisper-FineTuned-DL-Twi"   # base you're starting from
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load processor + base model
processor = WhisperProcessor.from_pretrained(MODEL_NAME, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
# optional: remove legacy forced_decoder_ids to avoid warnings
if hasattr(model, "generation_config") and hasattr(model.generation_config, "forced_decoder_ids"):
    try:
        model.generation_config.forced_decoder_ids = None
    except Exception:
        pass

# Move model to device AFTER PEFT wrap? either way is fine; we'll move to device after get_peft_model
# Configure LoRA
peft_config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",    # important for seq2seq models
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],  # good default for Whisper
    inference_mode=False,
)

model = get_peft_model(model, peft_config)
model.to(device)

# quick check of trainable params
model.print_trainable_parameters()


Device: cpu


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 38,055,552 || trainable%: 0.7750


In [ ]:
from datasets import DatasetDict

# ds = load_dataset("mozilla-foundation/common_voice_17_0", "tw", trust_remote_code=True)
ds = load_dataset("Ibaahjnr/Twi_Train_Dataset")

# First split train -> train + test (90/10)
split_ds = ds["train"].train_test_split(test_size=0.1, seed=42)
train_valid = split_ds["train"]
test_ds = split_ds["test"]

# Then split train_valid -> train + valid (88/12 overall)
split_ds2 = train_valid.train_test_split(test_size=0.111, seed=42)
train_ds = split_ds2["train"]
valid_ds = split_ds2["test"]

# Wrap back into DatasetDict
ds_splits = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds
})

# train_ds = ds["train"]
# valid_ds = ds["validation"]
# test_ds = ds["test"]
print("Train size:", len(train_ds))
print("Test size:", len(test_ds))
print("Valid size:", len(valid_ds))

print("Record: ", train_ds[0])

Train size: 22452
Test size: 2807
Valid size: 2804
Record:  {'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7f1a04ff11f0>, 'transcription': 'Yoo medaase'}


In [ ]:
# from dataclasses import dataclass
# from typing import Any, Dict, List, Union
# import torch

# class DataCollatorSpeechSeq2Seq:
#     processor: Any

#     def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
#         # Convert each input_feature into a tensor
#         input_features = [torch.tensor(f["input_features"]) for f in features]
#         batch_inputs = torch.stack(input_features)

#         # Convert labels into tensors & pad
#         labels = [torch.tensor(f["labels"]) for f in features]
#         batch_labels = torch.nn.utils.rnn.pad_sequence(
#             labels,
#             batch_first=True,
#             padding_value=self.processor.tokenizer.pad_token_id
#         )

#         return {
#             "input_features": batch_inputs,  # tensor
#             "labels": batch_labels           # tensor
#         }


# from dataclasses import dataclass
# from typing import Any, Dict, List, Union
# import torch

# class DataCollatorSpeechSeq2Seq:
#     processor: Any
#     decoder_start_token_id: int = None

#     def __call__(self, features: List[Dict[str, Union[torch.Tensor, Any]]]) -> Dict[str, torch.Tensor]:
#         # 1️⃣ Prepare input_features batch
#         input_features = [f["input_features"] for f in features]
#         batch_inputs = self.processor.feature_extractor.pad(
#             input_features,
#             padding=True,
#             return_tensors="pt"
#         ).input_features  # shape: [batch, seq_len, feature_dim]

#         # 2️⃣ Prepare labels batch
#         labels = [f["labels"] for f in features]
#         batch_labels = self.processor.tokenizer.pad(
#             {"input_ids": labels},
#             padding=True,
#             return_tensors="pt"
#         ).input_ids

#         # 3️⃣ Return dict
#         return {
#             "input_features": batch_inputs,
#             "labels": batch_labels
#         }


In [ ]:
import torch

class DataCollatorWhisper:
    def __init__(self, processor, decoder_start_token_id):
        self.processor = processor
        self.decoder_start_token_id = decoder_start_token_id

    def __call__(self, features):
        # Extract audio features (list of tensors -> tensor)
        input_features = [torch.tensor(f["input_features"]) for f in features]
        batch_inputs = torch.stack(input_features)

        # Extract labels (tokenized target text)
        labels = [torch.tensor(f["labels"]) for f in features]
        batch_labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=self.processor.tokenizer.pad_token_id
        )

        # Important: Replace padding with -100 so loss ignores them
        batch_labels[batch_labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "input_features": batch_inputs,   # <- Whisper expects this
            "labels": batch_labels,           # <- For teacher forcing
        }


In [ ]:
# quick smoke-test: take first 1000 examples
# train_ds = train_ds.select(range(min(1000, len(train_ds))))
# valid_ds = valid_ds.select(range(min(200, len(valid_ds))))


In [ ]:
from datasets import Audio
import torch
import torchaudio

# Make sure audio is decoded to array automatically
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# Preprocessing function
def prepare_batch(batch):
    # 1. Process audio into log-Mel features
    audio = batch["audio"]
    inputs = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"])
    batch["input_features"] = inputs["input_features"][0]

    # 2. Tokenize the target text (sentence)
    with processor.as_target_processor():
        labels = processor.tokenizer(batch["sentence"])
        batch["labels"] = labels["input_ids"]

    # Remove "input_ids" if processor.tokenizer created them
    if "input_ids" in batch:
        del batch["input_ids"]

    return batch

# def prepare_batch(batch):
    # 1. Resample
    # print(batch)
    # waveform = batch["audio"]["array"]
    # sr = batch["audio"]["sampling_rate"]

    # # Resample if needed
    # if sr != 16000:
    #     waveform = torchaudio.transforms.Resample(sr, 16000)(
    #         torch.tensor(waveform)
    #     ).numpy()

    # # Feature extraction
    # inputs = processor.feature_extractor(
    #     waveform,
    #     sampling_rate=16000,
    #     return_tensors="pt"
    # )

    # # Tokenize labels
    # labels = processor.tokenizer(
    #     batch["transcription"],  # or "transcription"
    #     return_tensors="pt"
    # ).input_ids

    # # Return tensors
    # batch["input_features"] = inputs.input_features.squeeze(0)
    # batch["labels"] = labels.squeeze(0)
    # return batch

# Apply mapping
# train_ds = train_ds.map(prepare_batch, remove_columns=train_ds.column_names)
# valid_ds = valid_ds.map(prepare_batch, remove_columns=valid_ds.column_names)
train_ds = train_ds.map(prepare_batch, remove_columns=train_ds.column_names)
valid_ds = valid_ds.map(prepare_batch, remove_columns=valid_ds.column_names)





Map:   0%|          | 0/22452 [00:00<?, ? examples/s]

AttributeError: 'WhisperProcessor' object has no attribute 'as_target_processor'

In [ ]:

# data_collator = DataCollatorSpeechSeq2Seq(processor=processor, model=model)
# data_collator = DataCollatorSpeechSeq2Seq(
#     processor=processor,
# )
data_collator = DataCollatorWhisper(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

# decoder_start_token_id=model.config.decoder_start_token_id
# print(data_collator)

wer = evaluate.load("wer")

def compute_metrics(pred):
    # predictions will be generated token IDs if predict_with_generate=True
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # sometimes predictions are logits if predict_with_generate=False — ensure they're IDs
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # decode preds
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    # replace -100 in labels as pad_token_id for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": wer.compute(predictions=pred_str, references=label_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-twi-lora",
    per_device_train_batch_size=4,   # lower if you hit OOM
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=2000,
    fp16=True,
    eval_strategy="steps",           # ✅ updated
    per_device_eval_batch_size=4,
    save_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    greater_is_better=False,
    metric_for_best_model="wer"
)
#     output_dir="./whisper-twi-lora",
#     per_device_train_batch_size=4,   # lower if OOM (2 or 1)
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=2,   # accum to simulate larger batch
#     learning_rate=1e-4,
#     num_train_epochs=3,
#     fp16=True,
#     eval_strategy="steps",
#     eval_steps=500,
#     save_steps=500,
#     logging_steps=50,
#     save_total_limit=3,
#     predict_with_generate=True,     # important for compute_metrics
#     load_best_model_at_end=True,
#     greater_is_better=False,
# )


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer.train()


TypeError: WhisperForConditionalGeneration.forward() got an unexpected keyword argument 'input_ids'

In [ ]:
# Save only the LoRA adapters (very small)
model.save_pretrained("./lora-adapters")
processor.save_pretrained("./lora-adapters-processor")

# If you want a single merged checkpoint (bigger ~ base model size):
from peft import PeftModel
# load base model again (clean base)
base = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
peft_model = PeftModel.from_pretrained(base, "./lora-adapters")
merged = peft_model.merge_and_unload()
merged.save_pretrained("./whisper-twi-merged")
processor.save_pretrained("./whisper-twi-merged")


In [ ]:
# push adapters or merged model
from huggingface_hub import HfApi
# ensure you're logged in (notebook_login earlier)
!git lfs install  # if needed

# Example: push merged model directory
from huggingface_hub import create_repo, upload_folder

create_repo(repo_id="your-username/whisper-twi-agri", private=False)
upload_folder(folder_path="./whisper-twi-merged", path_in_repo=".")
